# Word2Vec: Skip-Gram + 负采样 实现详解

## 核心思想

Word2Vec 的目标是将每个词映射为一个稠密向量（词向量），使得**语义相近的词在向量空间中距离也近**。

Skip-Gram 的训练方式是：给定一个**中心词**，预测其**上下文词**。模型通过大量这样的 (中心词, 上下文词) 对来学习词向量。

### 负采样（Negative Sampling）

原始 Skip-Gram 需要对整个词汇表做 softmax，计算量巨大。负采样的思路是：
- **正样本**：真实的 (中心词, 上下文词) 对，目标是让它们的点积尽可能大
- **负样本**：随机采样的噪声词对，目标是让它们的点积尽可能小

这样就把多分类问题转化为了二分类问题，大幅降低了计算复杂度。

## 1. 准备语料与构建词汇表

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np

In [2]:
# 原始语料
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the dog sleeps under the tree",
    "a quick brown fox runs fast"
]

# 分句、分词、转小写
sentences = [sentence.lower().split() for sentence in corpus]
print("分词结果:", sentences)

分词结果: [['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog'], ['the', 'dog', 'sleeps', 'under', 'the', 'tree'], ['a', 'quick', 'brown', 'fox', 'runs', 'fast']]


In [ ]:
# 统计每个词的出现次数
word_counts = Counter()
for sent in sentences:
    word_counts.update(sent)

print("词频统计:")
for word, count in word_counts.most_common():
    print(f"  {word}: {count}")

In [ ]:
# 构建词汇表：word <-> index 的双向映射
vocab = list(word_counts.keys())
vocab_size = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

print(f"词汇表大小: {vocab_size}")
print(f"word_to_idx: {word_to_idx}")

## 2. 生成训练样本对

Skip-Gram 的训练方式是：对于语料中的每个词，用它作为中心词，取其**窗口大小**内的相邻词作为上下文词。

例如，窗口大小为 2 时：
```
句子: "the quick brown fox"
中心词: 'brown' -> 上下文: ['the', 'quick', 'fox']
```

In [ ]:
# 生成 (中心词, 上下文词) 训练对
pairs = []
window = 2  # 窗口大小：中心词左右各取2个词

for sent in sentences:
    for i, center in enumerate(sent):
        # 取中心词左边 window 个词和右边 window 个词
        context_words = sent[max(0, i - window):i] + sent[i + 1:i + window + 1]
        for context in context_words:
            pairs.append((center, context))

print(f"共生成 {len(pairs)} 个训练对")
print(f"前10个训练对: {pairs[:10]}")

## 3. 负采样分布

负采样需要一个**噪声分布**来选择负样本。通常使用词频的 3/4 次方作为分布：

$$P(w_i) = \frac{f(w_i)^{3/4}}{\sum_j f(w_j)^{3/4}}$$

为什么要用 3/4 次方？
- 高频词（如 'the'）的概率被适当压低
- 低频词的概率被适当抬高
- 使得负采样分布更加均匀，避免高频词主导负样本

In [ ]:
# 构建负采样的概率分布（基于词频的3/4次方）
word_freq = np.array([word_counts[w] for w in vocab], dtype=np.float32)
word_freq_pow = word_freq ** 0.75
word_freq_dist = word_freq_pow / word_freq_pow.sum()  # 归一化为概率分布

print("原始词频:", dict(zip(vocab, word_freq.astype(int))))
print("\n3/4次方后的概率分布:")
for w, p in zip(vocab, word_freq_dist):
    print(f"  {w}: {p:.4f}")

## 4. Skip-Gram 模型定义

### 模型结构

Skip-Gram 有两套嵌入向量：
- `center_embed`：中心词的嵌入向量
- `context_embed`：上下文词的嵌入向量

最终的词向量通常取 `center_embed` 的权重。

### 损失函数

对于一个 (中心词, 上下文词) 对：
- **正样本损失**：$-\log\sigma(v_c \cdot v_o)$，其中 $v_c$ 是中心词向量，$v_o$ 是上下文词向量
- **负样本损失**：$-\sum_{k=1}^{K}\log\sigma(-v_c \cdot v_{n_k})$，其中 $v_{n_k}$ 是第 $k$ 个负样本的向量

总损失 = 正样本损失 + 负样本损失

In [ ]:
class SkipGramNeg(nn.Module):
    """
    Skip-Gram + 负采样模型
    
    两套嵌入向量：
    - center_embed: 中心词向量
    - context_embed: 上下文词向量
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # 两套独立的嵌入层
        self.center_embed = nn.Embedding(vocab_size, embed_dim)
        self.context_embed = nn.Embedding(vocab_size, embed_dim)
        # Xavier 初始化：保持梯度稳定
        nn.init.xavier_uniform_(self.center_embed.weight)
        nn.init.xavier_uniform_(self.context_embed.weight)
    
    def forward(self, center_idx, context_idx, neg_idx):
        """
        center_idx: (batch_size,)        中心词索引
        context_idx: (batch_size,)       正样本上下文词索引
        neg_idx: (batch_size, neg_num)   负样本词索引
        """
        # 获取嵌入向量
        center_emb = self.center_embed(center_idx)      # (B, D)
        context_emb = self.context_embed(context_idx)   # (B, D)
        neg_emb = self.context_embed(neg_idx)           # (B, neg_num, D)
        
        # === 正样本损失 ===
        # 中心词与上下文词的点积 -> sigmoid -> 取负对数
        # 目标：让正样本的点积尽量大
        pos_score = torch.sum(center_emb * context_emb, dim=1)  # (B,)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()
        
        # === 负样本损失 ===
        # 中心词与负样本词的点积 -> sigmoid -> 取负对数
        # 目标：让负样本的点积尽量小（即 sigmoid(-score) 尽量大）
        neg_score = torch.bmm(neg_emb, center_emb.unsqueeze(2)).squeeze(2)  # (B, neg_num)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-8).mean()
        
        return pos_loss + neg_loss
    
    def get_word_vector(self, word):
        """获取指定词的向量表示"""
        idx = torch.tensor([word_to_idx[word]])
        return self.center_embed(idx).detach().numpy().squeeze()

In [ ]:
# 可视化模型结构
print("模型结构:")
embed_dim = 50
model = SkipGramNeg(vocab_size, embed_dim)
print(model)
print(f"\n参数总量: {sum(p.numel() for p in model.parameters()):,}")

## 5. 训练过程

训练步骤：
1. 对每个 epoch，打乱训练对的顺序
2. 按 batch 取出 (中心词, 上下文词) 对
3. 对每个中心词，按噪声分布随机采样 `neg_num` 个负样本
4. 计算损失、反向传播、更新参数

In [ ]:
# 超参数
embed_dim = 50
neg_num = 5       # 每个正样本对应的负样本数量
batch_size = 8
num_epochs = 200
lr = 0.01

model = SkipGramNeg(vocab_size, embed_dim)
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
losses = []  # 记录每轮的平均损失

for epoch in range(num_epochs):
    total_loss = 0
    np.random.shuffle(pairs)
    
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        center_words, context_words = zip(*batch)
        
        # 转换为张量
        center_idx = torch.tensor([word_to_idx[w] for w in center_words])
        context_idx = torch.tensor([word_to_idx[w] for w in context_words])
        
        # 负采样：按噪声分布随机采样
        neg_samples = []
        for _ in range(len(center_words)):
            sampled = np.random.choice(vocab_size, size=neg_num, p=word_freq_dist)
            neg_samples.append(sampled)
        neg_idx = torch.tensor(neg_samples)
        
        # 前向传播 + 反向传播
        optimizer.zero_grad()
        loss = model(center_idx, context_idx, neg_idx)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(pairs)
    losses.append(avg_loss)
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {avg_loss:.4f}")

In [ ]:
# 绘制损失曲线
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Word2Vec Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 6. 使用词向量

训练完成后，每个词都有了一个 50 维的向量表示。我们可以通过**余弦相似度**来衡量两个词的语义相似度：

$$\text{cosine\_sim}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

值越接近 1，表示两个词越相似。

In [ ]:
# 获取词向量
print("'fox' 的前5维向量:", model.get_word_vector('fox')[:5])

# 余弦相似度
def cosine_sim(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

fox_vec = model.get_word_vector('fox')
dog_vec = model.get_word_vector('dog')
tree_vec = model.get_word_vector('tree')

print(f"\nfox <-> dog 的余弦相似度: {cosine_sim(fox_vec, dog_vec):.4f}")
print(f"fox <-> tree 的余弦相似度: {cosine_sim(fox_vec, tree_vec):.4f}")
print(f"dog <-> tree 的余弦相似度: {cosine_sim(dog_vec, tree_vec):.4f}")

In [ ]:
# 找出与 'fox' 最相似的词
fox_vec = model.get_word_vector('fox')
similarities = []
for w in vocab:
    if w == 'fox':
        continue
    vec = model.get_word_vector(w)
    sim = cosine_sim(fox_vec, vec)
    similarities.append((w, sim))

# 按相似度排序
similarities.sort(key=lambda x: x[1], reverse=True)
print("与 'fox' 最相似的词:")
for w, sim in similarities:
    print(f"  {w}: {sim:.4f}")

## 总结

| 概念 | 说明 |
|------|------|
| Skip-Gram | 给定中心词，预测上下文词 |
| 负采样 | 将多分类转为二分类，大幅降低计算量 |
| 3/4次方平滑 | 压低高频词、抬高低频词，使采样更均匀 |
| 两套嵌入 | center_embed 和 context_embed 分别学习，最终可取平均或只用 center |
| 余弦相似度 | 衡量两个词向量的方向相似性，值越接近1越相似 |